# 🚀 Optimización de Hiperparámetros: De los enfoques Básico a los basados en IA (Optuna)

En este notebook vamos a predecir el impago de tarjetas de crédito (Credit Default) utilizando un ensemble de **Gradient Boosting** para clasificación. Nuestro objetivo no es solo entrenar el modelo, sino descubrir cuál es la mejor forma de optimizar sus hiperparámetros para exprimir al máximo su rendimiento.

Pasaremos por 4 fases:
1. **Modelo Base:** Elegiremos hiperparámetros "al azar".
2. **Grid Search:** Probaremos combinaciones exhaustivas.
3. **Random Search:** Probaremos combinaciones aleatorias.
4. **Optuna:** Usaremos optimización bayesiana (búsqueda inteligente).

**IMPORTANTE**: Se recomienda cambiar el Entorno de Ejecución a GPU o TPU.

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 5.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import scipy.stats as stats
import optuna

# 1. Cargamos el Dataset: predicción de impagos de crédito en Taiwan (credit default)
print("Cargando dataset...")
credit_data = fetch_openml(data_id=42477, as_frame=True, parser='auto')

# Extraemos X e y. Forzamos 'y' a entero (0: Paga, 1: Impago)
X = credit_data.data
y = credit_data.target.astype(int)

# Codificación rápida (One-Hot Encoding) para evitar problemas con variables categóricas
X_encoded = pd.get_dummies(X, drop_first=True)

# 2. Partición (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)
print(f"Datos listos: {X_train.shape[0]} para entrenamiento, {X_test.shape[0]} para prueba.")

Cargando dataset...
Datos listos: 21000 para entrenamiento, 9000 para prueba.


---
### Fase 1: El Modelo Base (Hiperparámetros "al azar")
Vamos a configurar un modelo con hiperparámetros arbitrarios. Imaginemos que pensamos: *"Quiero muchos árboles (150), muy profundos (max_depth=8) y un aprendizaje rápido (learning_rate=0.5)"*. La receta perfecta para el sobreajuste (*overfitting*).

In [3]:
print("Entrenando Modelo Base...")
# Configuramos un modelo con parámetros "inventados"
base_model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=8,
    learning_rate=0.5,
    random_state=42
)

base_model.fit(X_train, y_train)

# Evaluamos usando ROC-AUC
preds_base = base_model.predict_proba(X_test)[:, 1]
auc_base = roc_auc_score(y_test, preds_base)

print(f"ROC-AUC del Modelo Base: {auc_base:.4f}")

Entrenando Modelo Base...
ROC-AUC del Modelo Base: 0.7293


---
### Fase 2: Búsqueda en Rejilla (GridSearchCV)
GridSearchCV es el enfoque de "fuerza bruta". Le damos una lista de valores y probará **todas las combinaciones posibles**.

Haremos una cuadrícula pequeña (solo 8 combinaciones) con una validación cruzada de 3 "folds" (`cv=3`). Eso significa que entrenará 24 modelos en total. Si pones demasiados valores, este proceso puede tardar horas.

In [4]:
print("Iniciando GridSearchCV...")

# Definimos una pequeña cuadrícula de opciones
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1 # Usar todos los procesadores
)

grid_search.fit(X_train, y_train)

print(f"Mejores Hiperparámetros (Grid): {grid_search.best_params_}")
print(f"Mejor ROC-AUC en validación: {grid_search.best_score_:.4f}")

# Evaluamos en Test
preds_grid = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
print(f"ROC-AUC en Test (Grid): {roc_auc_score(y_test, preds_grid):.4f}")

Iniciando GridSearchCV...
Mejores Hiperparámetros (Grid): {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}
Mejor ROC-AUC en validación: 0.7807
ROC-AUC en Test (Grid): 0.7781


---
### Fase 3: Búsqueda Aleatoria (RandomizedSearchCV)
En lugar de probar *todas* las combinaciones de una lista rígida, le pasamos **distribuciones estadísticas** y le pedimos que pruebe un número fijo de combinaciones aleatorias (`n_iter=10`).

A menudo, Random Search encuentra mejores modelos en menos tiempo que Grid Search porque explora un abanico más amplio de valores en lugar de quedarse atascado examinando una cuadrícula rígida.

In [5]:
print("Iniciando RandomizedSearchCV...")

# Definimos distribuciones en lugar de listas fijas
param_dist = {
    'n_estimators': stats.randint(50, 200),
    'max_depth': stats.randint(3, 8),
    'learning_rate': stats.uniform(0.01, 0.15) # Valores continuos entre 0.01 y 0.16
}

random_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10, # Solo probará 10 combinaciones aleatorias
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print(f"Mejores Hiperparámetros (Random): {random_search.best_params_}")
print(f"Mejor ROC-AUC en validación: {random_search.best_score_:.4f}")

# Evaluamos en Test
preds_random = random_search.best_estimator_.predict_proba(X_test)[:, 1]
print(f"ROC-AUC en Test (Random): {roc_auc_score(y_test, preds_random):.4f}")

Iniciando RandomizedSearchCV...
Mejores Hiperparámetros (Random): {'learning_rate': np.float64(0.033402796066365474), 'max_depth': 5, 'n_estimators': 124}
Mejor ROC-AUC en validación: 0.7808
ROC-AUC en Test (Random): 0.7782


---
### Fase 4: Búsqueda Inteligente (Optuna)
Grid Search es "ciego" (fuerza bruta). Random Search es "ciego" (aleatorio).

**Optuna tiene memoria.** Utiliza Optimización Bayesiana: en cada prueba, analiza los resultados de los modelos anteriores para deducir de forma inteligente qué hiperparámetros tienen más probabilidad de éxito en el siguiente intento.

Es el estándar de la industria hoy en día.

In [6]:
# Para silenciar un poco la salida detallada de Optuna en la libreta, que puede resultar excesiva
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # 1. Sugerir hiperparámetros de forma dinámica
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        # Usar escala logarítmica para el learning rate, una estrategia ideal para este parámetro
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    }

    # 2. Creamos y entrenamos el modelo con esos parámetros
    model = GradientBoostingClassifier(**params, random_state=42)

    # Hacemos una partición rápida del conjunto de train para validación interna
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    model.fit(X_tr, y_tr)

    # 3. Evaluamos
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

print("Iniciando Optuna (15 intentos)...")
# Creamos el "estudio" pidiendo que MAXIMICE la métrica (direction='maximize')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print(f"\nMejores Hiperparámetros (Optuna): {study.best_params}")
print(f"Mejor ROC-AUC en validación: {study.best_value:.4f}")

# Entrenamos el modelo final con la mejor configuración encontrada
best_optuna_model = GradientBoostingClassifier(**study.best_params, random_state=42)
best_optuna_model.fit(X_train, y_train)

# Evaluamos en Test
preds_optuna = best_optuna_model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC en Test (Optuna): {roc_auc_score(y_test, preds_optuna):.4f}")

Iniciando Optuna (15 intentos)...

Mejores Hiperparámetros (Optuna): {'n_estimators': 160, 'max_depth': 5, 'learning_rate': 0.033957202272310985}
Mejor ROC-AUC en validación: 0.7811
ROC-AUC en Test (Optuna): 0.7783
